In [2]:
get_ipython().system('pip install Gradio')
get_ipython().system('pip install Groq')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 1.2 MB/s eta 0:00:00


In [3]:
import gradio as gr
import os
from groq import Groq


In [4]:
client=Groq(api_key="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")

In [5]:
print("""
This AI-generated diet plan is for informational and educational purposes only.

• It is NOT a substitute for professional medical advice, diagnosis, or treatment.
• Always consult a qualified doctor, registered dietitian, or healthcare professional
  before making significant changes to your diet or lifestyle.
• If you have any medical conditions, allergies, are pregnant, or are taking
  medications, seek personalized medical guidance.
• Do not ignore or delay seeking professional medical advice based on this
  AI-generated recommendation.

This is for informational purposes only. For medical advice or diagnosis,
consult a qualified healthcare professional.
""")



This AI-generated diet plan is for informational and educational purposes only.

• It is NOT a substitute for professional medical advice, diagnosis, or treatment.
• Always consult a qualified doctor, registered dietitian, or healthcare professional
  before making significant changes to your diet or lifestyle.
• If you have any medical conditions, allergies, are pregnant, or are taking
  medications, seek personalized medical guidance.
• Do not ignore or delay seeking professional medical advice based on this
  AI-generated recommendation.

This is for informational purposes only. For medical advice or diagnosis,
consult a qualified healthcare professional.



In [6]:
def BMI_calculator(weight, height):
    height = height / 100
    bmi = weight / (height ** 2)
    return round(bmi, 2)


def BMI_category(bmi):

    if bmi < 18.5:
        return "Underweight"

    elif bmi < 25:
        return "Normal"

    elif bmi < 30:
        return "Overweight"

    else:
        return "Obese"

In [7]:
def BMR_calculator(gender, weight, height, age):

    if gender.upper() == "M":

        bmr = (
            10 * weight +
            6.25 * height -
            5 * age +
            5
        )

    else:

        bmr = (
            10 * weight +
            6.25 * height -
            5 * age -
            161
        )

    return round(bmr, 2)

In [8]:
def calculate_calories(bmr, activity_level):

    multipliers = {

        "Sedentary": 1.2,
        "Light": 1.375,
        "Moderate": 1.55,
        "Active": 1.725

    }

    return round(
        bmr * multipliers.get(activity_level, 1.2),   2
    )

In [9]:
def adjust_calories(calories, goal):

    goal = goal.lower()

    if "loose" in goal:

        return calories - 500

    elif "gain" in goal:

        return calories + 500

    else:

        return calories

In [10]:
def nutrition_rules(goal,activity,food_type,disease):
  rules=[]
  # Goal Based Rules
  if goal.lower() =="loose weight":
    rules.extend([
        "Avoid sugar and processed foods",
        "reduce the colories by 500/day",
        "Detox the body",
        "increase fiber intake"])
  elif goal.lower() =="gain weight":
    rules.extend([
        "Increase protein intake",
        "increase the colories by 500/day",
        "Eat 5-6 meals per day ",
        "Eat healthy Fats"])
  elif goal.lower() =="maintain weight":
    rules.extend([
        "Maintain balanced Calorie intake",
        "Eat Balanced meals ",
        "Limit junk food intake"])

  # Activity based rules
  if activity.lower() =="sedentary":
    rules.extend(["Keep carbohydrates intake moderate"])

  elif activity.lower() =="light":
    rules.extend(["Keep moderate carbohydrates and lean protein"])

  elif activity.lower() =="moderate":
     rules.extend(["Add complex carbohydrates"])

  elif activity.lower() =="active":
     rules.extend(["Increase protein Intake",
                   "Increase the colorie Intake",
                   "Maintain proper hydration " ])
  # food preference rules
  if food_type.lower()=="veg":
    rules.extend(["eat paneer,tofu,soya chunks,dal,sprouts,milk,curd as protein sources"])
  elif food_type.lower()=="non-veg":
    rules.extend(["Include eggs,chicken,fish and lean protein sources"])
  elif food_type.lower()=="jain":
    rules.extend(["Avoid onion,garlic, root vegetables", "tofu,soya chunks,dal,sprouts as protein"])

  # Discease Based Rules

  if disease.lower() =="diabetes":
    rules.extend(["choose low GI foods",
                  "avoid sugar and sweetened cold drinks",
                  "Eat small frequent meals",
                  "Avoid processed foods",
                  "increase dietary fibres"])

  elif disease.lower() =="PCOS":
    rules.extend(["choose low GI foods",
                  "avoid sugar and sweetened cold drinks",
                  "Add healthy fats",
                  "Avoid processed foods",
                  "Reduce Carbs"])

  elif disease.lower() =="Thyroid":
    rules.extend(["Ensure adquate iodine intake",
                  "consume selenium- rich foods ",
                  "Add healthy fats",
                  "Avoid processed foods",
                  "Reduce excess soya consumption"])

  elif disease.lower() =="Hypertension":
    rules.extend(["Limit sodium intake",
                  "consume potassium- rich foods ",
                  "Add healthy fats",
                  "Avoid processed foods",
                  "Drink more water"])

  elif disease.lower() =="Other":
    rules.extend(["Recommended consulting a physician"])

  elif disease.lower() =="None":
    rules.extend(["No specific dietary restrictions"])

  return rules

In [12]:
def generate_diet_plan(
    name,
    age,
    gender,
    height,
    weight,
    bmi,
    bmi_category,
    calories,
    rules,
    goal,
    activity,
    food_type,
    disease,
    region
):

    prompt = f"""
You are an experienced Indian Clinical Dietitian and Nutritionist.

Follow the latest ICMR-NIN Dietary Guidelines for Indians.

Create a personalized ONE-DAY diet plan.

=========================
USER DETAILS
=========================

Name : {name}

Age : {age}

Gender : {gender}

Height : {height} cm

Weight : {weight} kg

BMI : {bmi}

BMI Category : {bmi_category}

Target Calories : {calories}

Goal : {goal}

Activity Level : {activity}

Cuisine : {region}

Food Preference : {food_type}

Disease : {disease}

Nutrition Rules

{chr(10).join("- " + r for r in rules)}

=========================
OUTPUT FORMAT
=========================

Generate the response using Markdown.

Include the following sections.

# Health Summary

• BMI

• BMI Category

• Daily Calories

• Protein Requirement

• Water Intake

-----------------------------------

# Breakfast

Food

Serving Size

Calories

Protein

-----------------------------------

# Mid Morning Snack

-----------------------------------

# Lunch

-----------------------------------

# Evening Snack

-----------------------------------

# Dinner

-----------------------------------

# Bedtime Snack

(if required)

-----------------------------------

# Total Calories

-----------------------------------

# Macronutrients

Protein

Carbohydrates

Fat

-----------------------------------

# Foods To Avoid

-----------------------------------

# Recommended Foods

-----------------------------------

# Lifestyle Recommendations

-----------------------------------

# Exercise Recommendation

-----------------------------------

Rules

1. Use ONLY {region} cuisine.

2. Follow {food_type} diet.

3. Keep meals affordable.

4. Mention calories for every meal.

5. Mention serving sizes.

6. Modify meals according to {disease}.

7. Follow the nutrition rules provided.

8. Avoid foods restricted for the disease.

9. Give practical Indian food options.

10. Format the response neatly using Markdown tables where appropriate.
"""

    try:

        response = client.chat.completions.create(

            model="meta-llama/llama-4-scout-17b-16e-instruct",

            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ],

            temperature=0.7,

            max_completion_tokens=1800

        )

        return response.choices[0].message.content

    except Exception as e:

        return f"❌ Error generating diet plan.\n\n{str(e)}"

In [27]:
# Complete Pipeline
def generate_plan(
    name,
    age,
    height,
    weight,
    gender,
    goal,
    activity,
    region,
    food_type,
    disease
):

    # ------------------------
    # BMI
    # ------------------------

    bmi = BMI_calculator(weight, height)
    bmi_cat = BMI_category(bmi)

    # ------------------------
    # BMR
    # ------------------------

    bmr = BMR_calculator(
        gender,
        weight,
        height,
        age   )

    # ------------------------
    # Calories
    # ------------------------

    calories = calculate_calories(
        bmr,
        activity
    )

    calories = adjust_calories(
        calories,
        goal
    )

    # ------------------------
    # Nutrition Rules
    # ------------------------

    rules = nutrition_rules(
        goal,
        activity,
        food_type,
        disease
    )

    # ------------------------
    # AI Diet Plan
    # ------------------------

    diet = generate_diet_plan(
        name=name,
        age=age,
        gender=gender,
        height=height,
        weight=weight,
        bmi=bmi,
        bmi_category=bmi_cat,
        calories=calories,
        rules=rules,
        goal=goal,
        activity=activity,
        food_type=food_type,
        disease=disease,
        region=region
    )

    # ------------------------
    # Final Output
    # ------------------------

    output = f"""
# 🥗 AI Diet Planner

## User Summary

**Name:** {name}

**Age:** {age}

**Gender:** {gender}

**Height:** {height} cm

**Weight:** {weight} kg

**BMI:** {bmi}

**BMI Category:** {bmi_cat}

**Target Calories:** {calories}

---

{diet}

---

{DISCLAIMER}
"""

    return output

In [14]:
# Interface of Gradio

In [28]:
import gradio as gr

demo = gr.Interface(
    fn=generate_plan,
    inputs=[
        gr.Textbox(label="Name"),
        gr.Number(label="Age"),
        gr.Number(label="Height (cm)"),
        gr.Number(label="Weight (kg)"),
        gr.Radio(["Male", "Female"], label="Gender"),
        gr.Dropdown(
            ["Loose Weight", "Maintain Weight", "Gain Weight"],
            label="Goal"
        ),
        gr.Dropdown(
            ["Sedentary", "Light", "Moderate", "Active"],
            label="Activity Level"
        ),
        gr.Dropdown(
            ["Maharashtrian", "Punjabi", "Gujarati", "South Indian", "Bengali"],
            label="Region"
        ),
        gr.Radio(
            ["Veg", "Non-veg", "Jain"],
            label="Food Type"
        ),
        gr.Dropdown(
            ["Diabetes", "PCOS", "Thyroid", "Hypertension", "Other", "None"],
            label="Disease"
        )
    ],
    outputs=gr.Markdown(),
    title="🥗 AI Diet Planner",
    description="""
Generate a personalized Indian diet plan using AI.

The application:
- Calculates BMI
- Calculates BMR
- Calculates Daily Calories
- Applies Nutrition Rules
- Uses Groq Llama 4 to generate a personalized Indian diet plan
""",
    theme=gr.themes.Soft()
)

if __name__ == "__main__":
    demo.launch(
        server_name="0.0.0.0",
        server_port=7860,
        share=False
    )

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

In [29]:
pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 55.8 MB/s eta 0:00:00


In [30]:
content = """gradio
groq
python-dotenv
pandas
numpy
scikit-learn
"""

with open("requirements.txt", "w") as f:
    f.write(content)

print("requirements.txt created successfully!")

requirements.txt created successfully!


In [31]:
!cat requirements.txt

gradio
groq
python-dotenv
pandas
numpy
scikit-learn
